# LNS-Triton ibm01 — Generate GIF
Captures a frame at every accepted LNS improvement and assembles a GIF.

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip())
assert result.returncode == 0, 'No GPU detected!'

In [ ]:
import os
if os.path.isdir('/content/macro-place-challenge'):
    !cd /content/macro-place-challenge && git fetch && git reset --hard origin/main
else:
    !git clone https://github.com/rkothari3/macro-place-challenge.git /content/macro-place-challenge

%cd /content/macro-place-challenge
!git log --oneline -1

In [ ]:
!git submodule update --init external/MacroPlacement

In [ ]:
!pip install -e . --quiet
!pip install igraph imageio --quiet
import torch
print('torch:', torch.__version__, '  cuda:', torch.cuda.is_available())

In [ ]:
%cd /content/macro-place-challenge/submissions/analytical_placer/density_ext
!pip install -e . --quiet
%cd /content/macro-place-challenge
print('density_ext OK')

## Frame capture helpers

In [ ]:
import sys, importlib.util, io, time
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
import imageio.v2 as imageio
from pathlib import Path

from macro_place.loader import load_benchmark_from_dir
from macro_place.objective import _set_placement

BENCH = 'ibm01'
ICCAD = Path('/content/macro-place-challenge/external/MacroPlacement/Testcases/ICCAD04')
b, plc = load_benchmark_from_dir(str(ICCAD / BENCH))
print(f'{BENCH}: {b.num_hard_macros} hard macros, {b.num_soft_macros} soft macros')

In [ ]:
def render_frame(pos, b, title='', proxy=None):
    """Render a single placement frame. Returns RGB numpy array."""
    fig, ax = plt.subplots(figsize=(6, 6), dpi=120)
    ax.set_xlim(0, b.canvas_width)
    ax.set_ylim(0, b.canvas_height)
    ax.set_aspect('equal')
    ax.set_facecolor('#f8f8f8')
    fig.patch.set_facecolor('#ffffff')

    # Draw canvas border
    ax.add_patch(Rectangle((0, 0), b.canvas_width, b.canvas_height,
                            fill=False, edgecolor='black', linewidth=1.5))

    pos_np = pos.cpu().numpy()
    sizes_np = b.macro_sizes.cpu().numpy()

    # Soft macros (draw first, behind)
    for i in range(b.num_hard_macros, b.num_macros):
        x, y = pos_np[i]
        w, h = sizes_np[i]
        ax.add_patch(Rectangle((x - w/2, y - h/2), w, h,
                                facecolor='#b0c4de', alpha=0.2,
                                edgecolor='#888', linewidth=0.3, linestyle='--'))

    # Hard macros
    for i in range(b.num_hard_macros):
        x, y = pos_np[i]
        w, h = sizes_np[i]
        fixed = b.macro_fixed[i].item()
        fc = '#cc3333' if fixed else '#3366cc'
        ax.add_patch(Rectangle((x - w/2, y - h/2), w, h,
                                facecolor=fc, alpha=0.7,
                                edgecolor='#111', linewidth=0.4))

    # I/O ports
    if b.port_positions.shape[0] > 0:
        pp = b.port_positions.cpu().numpy()
        ax.scatter(pp[:, 0], pp[:, 1], s=6, c='#228b22', zorder=5)

    label = title
    if proxy is not None:
        label += f'  proxy={proxy:.4f}'
    ax.set_title(label, fontsize=10, pad=4)
    ax.set_xlabel('x (μm)', fontsize=8)
    ax.set_ylabel('y (μm)', fontsize=8)
    ax.tick_params(labelsize=7)

    legend = [
        mpatches.Patch(facecolor='#3366cc', alpha=0.7, label='Hard macro'),
        mpatches.Patch(facecolor='#cc3333', alpha=0.7, label='Fixed macro'),
    ]
    ax.legend(handles=legend, loc='upper right', fontsize=7)

    fig.tight_layout(pad=0.5)
    buf = io.BytesIO()
    fig.savefig(buf, format='png')
    plt.close(fig)
    buf.seek(0)
    return imageio.imread(buf)

print('render_frame OK')

## Run analytical warm start — capture initial frame

In [ ]:
# Load placer modules
SUBMISSIONS = Path('/content/macro-place-challenge/submissions')
sys.path.insert(0, str(SUBMISSIONS / 'analytical_placer'))
sys.path.insert(0, str(SUBMISSIONS / 'lns_triton_placer'))

_spec = importlib.util.spec_from_file_location('_ap', str(SUBMISSIONS / 'analytical_placer/placer.py'))
_ap = importlib.util.module_from_spec(_spec)
sys.modules['_ap'] = _ap
_spec.loader.exec_module(_ap)

from triton_ops import hv_demand_triton
print('modules loaded')

In [ ]:
frames = []
frame_labels = []

device = torch.device('cuda')

# Best-of-3 warm start (same as placer.py)
best_warm, best_proxy = None, float('inf')
for i in range(3):
    pos = _ap.AnalyticalPlacer().place(b)
    _set_placement(plc, pos, b)
    proxy = float(plc.get_cost()) + 0.5*float(plc.get_density_cost()) + 0.5*float(plc.get_congestion_cost())
    print(f'  warm start {i+1}/3: proxy={proxy:.4f}')
    if proxy < best_proxy:
        best_proxy, best_warm = proxy, pos.clone()

print(f'Best warm start: {best_proxy:.4f}')
frames.append(render_frame(best_warm, b, title='Analytical warm start', proxy=best_proxy))
frame_labels.append(f'warm  {best_proxy:.4f}')

## Run LNS — capture a frame at every accepted improvement

In [ ]:
from macro_place.objective import _set_placement

# Pull LNS helpers from lns_triton_placer
_lns_spec = importlib.util.spec_from_file_location('_lns', str(SUBMISSIONS / 'lns_triton_placer/placer.py'))
_lns = importlib.util.module_from_spec(_lns_spec)
sys.modules['_lns'] = _lns
_lns_spec.loader.exec_module(_lns)

_preprocess               = _lns._preprocess
_select_neighborhood_by_peak_reduction = _lns._select_neighborhood_by_peak_reduction
_gradient_refine_subset   = _lns._gradient_refine_subset
_legalize                 = _lns._legalize
macro_overlap_loss        = _lns.macro_overlap_loss

def _true_proxy(pos, b, plc):
    _set_placement(plc, pos.cpu(), b)
    return float(plc.get_cost()) + 0.5*float(plc.get_density_cost()) + 0.5*float(plc.get_congestion_cost())

data = _preprocess(b, device)
print('LNS helpers ready')

In [ ]:
LNS_BUDGET   = 600.0   # 10 min — enough to get a good set of improvement frames
K            = 30
INNER_STEPS  = 30
NO_IMP_LIMIT = 50

movable_idx = b.get_movable_mask().nonzero(as_tuple=True)[0]
best_pos    = best_warm.clone()
best_proxy  = _true_proxy(best_pos, b, plc)

t0 = time.time()
iteration, no_imp = 0, 0

while True:
    elapsed = time.time() - t0
    if elapsed >= LNS_BUDGET or no_imp >= NO_IMP_LIMIT:
        break

    subset = _select_neighborhood_by_peak_reduction(
        best_pos, movable_idx, b, data, device, k=K, budget=5)

    candidate = _gradient_refine_subset(
        best_pos, subset, b, data, device, steps=INNER_STEPS, cong_w=0.6)

    candidate = _legalize(candidate, b, time_budget_s=5.0, max_passes=100, verbose=False)

    with torch.no_grad():
        ovl = macro_overlap_loss(candidate.to(device), b.macro_sizes.to(device),
                                  b.num_hard_macros, gap=0.0)
    if ovl.item() > 1e-6:
        no_imp += 1; iteration += 1; continue

    proxy = _true_proxy(candidate, b, plc)
    if proxy < best_proxy:
        improvement = best_proxy - proxy
        best_proxy  = proxy
        best_pos    = candidate.clone()
        no_imp      = 0
        print(f'iter {iteration:4d}  proxy={proxy:.4f}  Δ={improvement:+.5f}  t={elapsed:.0f}s')
        frames.append(render_frame(best_pos, b,
                                   title=f'LNS iter {iteration}', proxy=proxy))
        frame_labels.append(f'iter {iteration}  {proxy:.4f}')
    else:
        no_imp += 1

    iteration += 1

print(f'\nDone: {iteration} iters, {len(frames)} frames captured, best proxy={best_proxy:.4f}')

## Assemble GIF

In [ ]:
# Hold first and last frame longer
durations = [1.5] + [0.4] * (len(frames) - 2) + [3.0] if len(frames) > 2 else [2.0] * len(frames)

OUT = '/content/macro-place-challenge/assets/lns_ibm01.gif'
imageio.mimsave(OUT, frames, duration=durations, loop=0)
print(f'Saved: {OUT}  ({len(frames)} frames)')

In [ ]:
# Preview in notebook
from IPython.display import Image
Image(OUT)